# Overview
This notebook explores practical applications of MetaDoE through a hyperparameter tuning case study.

In [1]:
include("../../src/MetaDoE.jl")
using .MetaDoE: Experiments, ConstraintEnforcement, Constraints, PSO, Objectives, Designs, Models
using Base.Iterators
using LinearAlgebra
using NPZ
using Optim

# Optimize Design

In [2]:
N = 20
K = 10

# Lower
lower_A = -I(K)
lower_b = [70 -500 -100 -10 0 0 0 -8 -1 -8]

# Upper
upper_A = I(K)
upper_b = [-20 1000 500 200 5 1 0.7 64 4 64]

# Factor constraints 
factor_A = [0   0  -1  10   0   0   0   0   0   0;
            0   0   0   0   0   0   0  -1   0   1]
factor_b = [0 0]

# Combined
A = Array{Float64}(vcat(lower_A, upper_A))
b = vec(hcat(lower_b, upper_b))

model = Models.quadratic
experiment = Experiments.create(N, K, model)
experiment = Experiments.with_linear_constraints(experiment, A, b)

Main.MetaDoE.Experiments.Experiment(Dict("8" => 8, "4" => 4, "1" => 1, "5" => 5, "2" => 2, "6" => 6, "7" => 7, "10" => 10, "9" => 9, "3" => 3…), Main.MetaDoE.ConstraintEnforcement.LinearConstraints([-1.0 0.0 … 0.0 0.0; 0.0 -1.0 … 0.0 0.0; … ; 0.0 0.0 … 1.0 0.0; 0.0 0.0 … 0.0 1.0], [70.0, -500.0, -100.0, -10.0, 0.0, 0.0, 0.0, -8.0, -1.0, -8.0, -20.0, 1000.0, 500.0, 200.0, 5.0, 1.0, 0.7, 64.0, 4.0, 64.0]), Main.MetaDoE.Models.var"#model_builder#19"{Int64, Int64, Bool, Vector{Any}, Bool, Bool}(2, 0, true, Any[], false, true), 20, 10)

In [3]:
runner_params = PSO.runner_params(200, 200, 1e-6)
pso_params = PSO.create_hyperparams(500)

context = PSO.create_context(
    experiment, 
    Objectives.D; 
    hyperparams = pso_params,
    runner_params = runner_params,
    enforcer_type = ConstraintEnforcement.Parametric
)

runner_state, history = PSO.optimize(context)

Iteration: 0 Best score: -64.98782706087837
Iteration: 1 Best score: -77.86839877720905
Iteration: 2 Best score: -83.32504307386485
Iteration: 3 Best score: -83.32676946685123
Iteration: 4 Best score: -83.32676946685123
Iteration: 5 Best score: -83.32676946685123
Iteration: 6 Best score: -83.32676946685123
Iteration: 7 Best score: -83.32676946685123
Iteration: 8 Best score: -83.32676946685123
Iteration: 9 Best score: -83.32676946685123
Iteration: 10 Best score: -83.32676946685123
Iteration: 11 Best score: -83.32676946685123
Iteration: 12 Best score: -83.32676946685123
Iteration: 13 Best score: -83.32676946685123
Iteration: 14 Best score: -84.3173899222873
Iteration: 15 Best score: -84.3173899222873
Iteration: 16 Best score: -84.3173899222873
Iteration: 17 Best score: -84.3173899222873
Iteration: 18 Best score: -84.3173899222873
Iteration: 19 Best score: -84.3173899222873
Iteration: 20 Best score: -84.3173899222873
Iteration: 21 Best score: -84.3173899222873
Iteration: 22 Best score: -8

(Main.MetaDoE.PSO.RunnerState(Main.MetaDoE.PSO.Swarm(Main.MetaDoE.PSO.ParticleState([-46.721440644894926 -20.28367793982812 … -49.28252321121262 -40.99680710775378; -32.37365806674304 -20.0 … -22.037887317002838 -50.82096982679242; … ; -34.994291333711004 -28.9222786639303 … -37.18977263646364 -23.92014311175065; -20.12127416543676 -21.065680020356215 … -50.55140008353779 -44.35278226198962;;; 537.4663859849519 504.1610592629034 … 532.2900631284608 955.6503737651468; 500.0 513.1602660660658 … 807.7097127022201 558.0947560098377; … ; 549.7623852301696 806.3767506263544 … 710.8291944069435 758.7712623186723; 500.0 545.7053808146978 … 505.6799096423891 755.5575260973699;;; 206.6272603723216 100.0 … 100.0 165.1986654744847; 160.7950039885387 128.09242444415662 … 149.89585129865645 321.1385462562534; … ; 151.81415651488265 163.64560275235976 … 100.0 140.47724264359107; 101.05456111117074 246.2853101123528 … 100.0 210.6778313295677;;; 143.5777257355059 74.72991127780388 … 93.35258869738529 1

In [4]:
PSO.save_results(runner_state; location = "hyperparameter_tuning_case_study_2.npy")

In [5]:
PSO.save_results(runner_state; location = "hyperparameter_tuning_case_study.npy")

# Fit Model

In [6]:
losses = npzread("case_study_responses.npy")
settings = npzread("hyperparameter_tuning_case_study.npy")

LoadError: SystemError: opening file "case_study_responses.npy": No such file or directory

In [ ]:
model = Models.quadratic
F = model(settings)
β = F \ losses

21-element Vector{Float64}:
  7.863710156998575
  0.5232611886101487
  0.005256326738197957
 -0.009535172124199513
  8.560548045075197e-6
  0.003784669423712553
 -1.096860819055528e-5
 -0.028333049326344693
  0.0001415929180390749
  1.302164265884832
 -0.5332593804608906
 -5.87220266208906
  2.052017501864995
  7.128734005046965
 -5.936695688697819
 -0.08459447961840699
  0.0002574318539912251
  8.48729352283256
 -1.6838023344198507
  0.07464218866646075
 -0.0002910416359778379

# Obtain Loss-Minimizing Hyperparameters

In [ ]:
function surrogate_loss(x)
    return model(x)' * β
end

K = 10

# Use Fminbox to enforce box constraints
result = optimize(
    surrogate_loss,                  
    -lower_b, upper_b,               
    (-lower_b .+ upper_b) ./ 2,      
    Fminbox(LBFGS())
)

x_opt = result.minimizer